# Multimodal SFT: Video Action Recognition

Train GPT-4.1 on UCF101 clips using a deterministic train/validation split and three ordered frames per clip. Each color frame receives a 201 by 201 Gaussian blur at its original resolution and is JPEG-encoded without edge extraction or timestamp overlays. The notebook monitors training and downloads service-emitted loss and token-accuracy metrics.

Actual results may vary by model version, data, configuration, region availability, and service conditions.

## Setup

Copy `../.env.template` to `../.env`, set `FOUNDRY_PROJECT_ENDPOINT`, run `az login`, and install `../../../requirements.lock`. Generated data and the duplicate-safe run state remain under ignored `outputs/`.

Preparation cells make no Foundry calls. Execute the client and training cells to upload and submit or resume a paid job. Job creation disables SDK retries.

In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import os
import time
from io import StringIO
from pathlib import Path

import cv2
import kagglehub
import numpy as np
import pandas as pd
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip()
MODEL = "gpt-4.1-2025-04-14"
SEED = 0
FRAMES_PER_VIDEO = 3
TRAIN_PER_CLASS = 12
VALIDATION_PER_CLASS = 4
OUTPUT_DIR = ROOT / "outputs" / "video-action-recognition"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STATE_PATH = OUTPUT_DIR / "run-state.json"

credential = None
project_client = None
client = None
print({"model": MODEL, "training_type": "GlobalStandard", "project_endpoint": PROJECT_ENDPOINT})

## Prepare the exact source-demo training split

The source demo uses Kaggle dataset `matthewjansen/ucf101-action-recognition`, its official `train.csv` and `val.csv`, the three most common training labels, seed 0, 12 training clips and 4 validation clips per class, and three evenly spaced frames per clip.

In [ ]:
os.environ["KAGGLEHUB_CACHE"] = str(ROOT / "data" / "downloads" / "kagglehub")
dataset_path = Path(kagglehub.dataset_download("matthewjansen/ucf101-action-recognition"))

train_source = pd.read_csv(dataset_path / "train.csv")
validation_source = pd.read_csv(dataset_path / "val.csv")
for frame in (train_source, validation_source):
    frame["clip_path"] = frame["clip_path"].str.lstrip("/")

top_classes = train_source["label"].value_counts().index[:3]

def stratified_sample(frame: pd.DataFrame, samples_per_class: int) -> pd.DataFrame:
    return (
        frame[frame["label"].isin(top_classes)]
        .groupby("label", group_keys=False)
        .sample(n=samples_per_class, random_state=SEED)
        .reset_index(drop=True)
    )

train_frame = stratified_sample(train_source, TRAIN_PER_CLASS)
validation_frame = stratified_sample(validation_source, VALIDATION_PER_CLASS)
assert set(train_frame["clip_path"]).isdisjoint(validation_frame["clip_path"])

display(pd.DataFrame({
    "training": train_frame["label"].value_counts(),
    "validation": validation_frame["label"].value_counts(),
}).fillna(0).astype(int))

## Encode ordered video frames

Extract three evenly spaced color frames from each clip, apply a 201 by 201 Gaussian blur at the original resolution, and JPEG-encode them in temporal order. The transformation preserves clip selection, labels, splits, and frame positions.


In [ ]:
def extract_blurred_frames(video_path: Path, count: int) -> list[str]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Error opening video file: {video_path}")
    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if count <= 0 or count > frame_count:
        capture.release()
        raise ValueError(f"Invalid frame count {count} for {video_path}")
    duration = frame_count / fps
    frame_indices = (np.linspace(0, duration, count, endpoint=False) * fps).astype(int)
    encoded_frames = []
    for frame_index in frame_indices:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = capture.read()
        if not ok:
            continue
        smoothed = cv2.GaussianBlur(frame, (201, 201), 0)
        ok, buffer = cv2.imencode(".jpg", smoothed)
        if not ok:
            capture.release()
            raise ValueError(f"JPEG encoding failed for {video_path}")
        encoded_frames.append(base64.b64encode(buffer).decode("utf-8"))
    capture.release()
    if len(encoded_frames) != count:
        raise ValueError(f"Extracted {len(encoded_frames)} of {count} frames from {video_path}")
    return encoded_frames

top_classes_str = ", ".join(top_classes)
system_message = f"""You are an expert in identifying human activities in video clips.
You are provided with a time-ordered series of smoothly blurred color frames from a video.
Use the objects, poses, and motion progression across the ordered frames to identify
one activity from this list:
{top_classes_str}

Provide the result as a valid JSON object in the exact format below:
{{
    "activity": "The identified activity from the provided list, spelled exactly as given."
}}
"""

def build_rows(frame: pd.DataFrame) -> list[dict]:
    rows = []
    for sample in frame.itertuples(index=False):
        images = [
            {"type": "image_url", "image_url": {"url": f"data:image/jpg;base64,{value}"}}
            for value in extract_blurred_frames(dataset_path / sample.clip_path, FRAMES_PER_VIDEO)
        ]
        rows.append({"messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": "These are the frames from the video."},
            {"role": "user", "content": images},
            {"role": "assistant", "content": json.dumps({"activity": sample.label})},
        ]})
    return rows

def write_jsonl(rows: list[dict], path: Path) -> Path:
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows:
            stream.write(json.dumps(row) + "\n")
    return path

def file_summary(path: Path) -> dict:
    payload = path.read_bytes()
    return {
        "path": str(path),
        "rows": sum(bool(line.strip()) for line in payload.splitlines()),
        "bytes": len(payload),
        "sha256": hashlib.sha256(payload).hexdigest(),
    }

train_path = write_jsonl(build_rows(train_frame), OUTPUT_DIR / "ucf101-train.jsonl")
validation_path = write_jsonl(build_rows(validation_frame), OUTPUT_DIR / "ucf101-validation.jsonl")
input_summary = {"training": file_summary(train_path), "validation": file_summary(validation_path)}
(OUTPUT_DIR / "lineage.json").write_text(json.dumps({
    "dataset": "matthewjansen/ucf101-action-recognition",
    "dataset_version": "4",
    "classes": list(top_classes),
    "seed": SEED,
    "frames_per_video": FRAMES_PER_VIDEO,
    "source_frame_positions": "linspace(0, duration, 3, endpoint=False) * fps",
    "frame_representation": "color frames, full-resolution Gaussian blur 201x201, OpenCV default JPEG, no timestamp strip",
    "files": input_summary,
}, indent=2), encoding="utf-8")
display(pd.DataFrame(input_summary).T)


## Submit once and monitor to terminal

The ignored run-state file is written immediately after submission. Rerunning the notebook retrieves that job instead of creating a duplicate, and refuses to reuse state if the project, model, or generated file hashes differ.

In [ ]:
if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in .env before running training.")
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project_client.get_openai_client()


In [ ]:
TERMINAL_STATES = {"succeeded", "failed", "cancelled"}

def retry_service_call(operation, name: str, attempts: int = 6):
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except Exception as error:
            if attempt == attempts:
                raise RuntimeError(f"{name} failed after {attempts} attempts") from error
            delay = min(2 ** attempt, 30)
            print({"operation": name, "retry": attempt, "delay_seconds": delay})
            time.sleep(delay)


def wait_for_file(file_id: str, timeout_seconds: int = 900):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        uploaded = retry_service_call(lambda: client.files.retrieve(file_id), "retrieve uploaded file")
        print({"file_id": file_id, "status": uploaded.status})
        if uploaded.status == "processed":
            return uploaded
        if uploaded.status in {"error", "failed", "cancelled", "expired"}:
            raise RuntimeError(f"File {file_id} ended as {uploaded.status}: {uploaded}")
        time.sleep(10)
    raise TimeoutError(f"Timed out waiting for file {file_id}")

job = None
signature = {
    "project_endpoint": PROJECT_ENDPOINT,
    "model": MODEL,
    "training_sha256": input_summary["training"]["sha256"],
    "validation_sha256": input_summary["validation"]["sha256"],
}

if RUN_STATE_PATH.exists():
    run_state = json.loads(RUN_STATE_PATH.read_text(encoding="utf-8"))
    for key, expected in signature.items():
        if run_state.get(key) != expected:
            raise RuntimeError(f"Run-state mismatch for {key}; remove {RUN_STATE_PATH} only to start a fresh job.")
    if run_state.get("submission_pending") and not run_state.get("job_id"):
        raise RuntimeError("Previous submission outcome is unknown; reconcile the remote job and record its ID before retrying. Do not clear pending state to resubmit.")
    job_id = run_state["job_id"]
    print({"action": "reuse_recorded_job", "job_id": job_id})
else:
    with train_path.open("rb") as stream:
        training_file = client.files.create(file=stream, purpose="fine-tune")
    with validation_path.open("rb") as stream:
        validation_file = client.files.create(file=stream, purpose="fine-tune")
    training_file = wait_for_file(training_file.id)
    validation_file = wait_for_file(validation_file.id)
    run_state = {
        **signature,
        "training_file_id": training_file.id,
        "validation_file_id": validation_file.id,
        "submission_pending": True,
    }
    RUN_STATE_PATH.write_text(json.dumps(run_state, indent=2), encoding="utf-8")
    job = client.with_options(max_retries=0).fine_tuning.jobs.create(
        suffix="video-action-ucf101-blur201",
        training_file=training_file.id,
        validation_file=validation_file.id,
        model=MODEL,
        seed=SEED,
        method={"type": "supervised", "supervised": {"hyperparameters": {"n_epochs": 1}}},
        extra_body={"trainingType": "GlobalStandard"},
    )
    job_id = job.id
    run_state = {
        **signature,
        "job_id": job_id,
        "submission_pending": False,
        "training_file_id": training_file.id,
        "validation_file_id": validation_file.id,
        "submitted_at": pd.Timestamp.now("UTC").isoformat(),
    }
    RUN_STATE_PATH.write_text(json.dumps(run_state, indent=2), encoding="utf-8")
    print({"action": "submitted", "job_id": job_id})

status_history = []
last_status = None
while True:
    job = retry_service_call(lambda: client.fine_tuning.jobs.retrieve(job_id), "retrieve fine-tuning job")
    if job.status != last_status:
        status_history.append({"time": pd.Timestamp.now("UTC").isoformat(), "status": job.status})
        print({"job_id": job_id, "status": job.status})
        last_status = job.status
    if job.status in TERMINAL_STATES:
        break
    time.sleep(30)

job_payload = job.to_dict()
events_page = retry_service_call(
    lambda: client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=100),
    "list fine-tuning events",
)
events = [event.to_dict() for event in events_page.data]
(OUTPUT_DIR / "job-final.json").write_text(json.dumps(job_payload, indent=2, default=str), encoding="utf-8")
(OUTPUT_DIR / "job-events.json").write_text(json.dumps(events, indent=2, default=str), encoding="utf-8")
run_state.update({"status": job.status, "status_history": status_history})
RUN_STATE_PATH.write_text(json.dumps(run_state, indent=2), encoding="utf-8")

preprocessing_events = [event for event in events if "Preprocessing Summary" in event.get("message", "")]
if preprocessing_events:
    display(pd.DataFrame(preprocessing_events)[["created_at", "level", "message"]])
print({"job_id": job_id, "status": job.status, "error": job_payload.get("error")})


## Download service training metrics

Only service-emitted training or validation loss and token-accuracy fields are displayed. If the job is not successful or emits no result file, the terminal job and preprocessing events remain the authoritative evidence.

In [ ]:
if job is not None:
    if job.status == "succeeded" and job_payload.get("result_files"):
        result_file_id = job_payload["result_files"][0]
        metrics_text = client.files.content(result_file_id).content.decode("utf-8")
        metrics = pd.read_csv(StringIO(metrics_text))
        metrics.to_csv(OUTPUT_DIR / "training-metrics.csv", index=False)
        metric_columns = [
            column for column in metrics.columns
            if column == "step" or "loss" in column or "token_accuracy" in column
        ]
        display(metrics[metric_columns])
    else:
        print("No service training metrics were emitted for this terminal job.")


    if job.status != "succeeded":
        raise RuntimeError(f"Fine-tuning job {job_id} ended in {job.status}: {job_payload.get('error')}")
if client is not None:
    client.close()
if project_client is not None:
    project_client.close()
if credential is not None:
    credential.close()


## Result

Use the terminal job record, service preprocessing events, and emitted loss/token-accuracy metrics as the complete outcome of this training-only workflow.